In [1]:
import nltk
from nltk import CFG

# 定义句法规则
grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the' | 'a'
    N -> 'cat' | 'dog'
    V -> 'chased'
""")

# 创建解析器
parser = nltk.ChartParser(grammar)

sentence = ['the', 'cat', 'chased', 'a', 'dog']

# 解析句子并显示句法树
for tree in parser.parse(sentence):
    tree.pretty_print()

              S               
      ________|_____           
     |              VP        
     |         _____|___       
     NP       |         NP    
  ___|___     |      ___|___   
Det      N    V    Det      N 
 |       |    |     |       |  
the     cat chased  a      dog



In [3]:
from transformers import BertTokenizer, BertModel
import torch

# 加载BERT的预训练模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 输入句子
sentence = "The cat chased a dog."
inputs = tokenizer(sentence, return_tensors="pt")

# 使用BERT生成嵌入
outputs = model(**inputs)

# 获取BERT的输出层中的词嵌入
embeddings = outputs.last_hidden_state

print(embeddings.shape)  # 查看词嵌入的形状

C:\Users\13979\anaconda3\envs\exercise\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([1, 8, 768])


In [5]:
from sklearn.metrics import f1_score

# 模拟一些人工标注的句法树（例如，0表示不匹配，1表示匹配）
y_true = [1, 1, 0, 1, 0]  # 假设的真实标签
y_pred_no_bert = [1, 0, 0, 1, 0]  # 传统句法树分析结果
y_pred_with_bert = [1, 1, 0, 1, 1]  # 使用BERT的句法分析结果

# 计算F1-Score
f1_no_bert = f1_score(y_true, y_pred_no_bert)
f1_with_bert = f1_score(y_true, y_pred_with_bert)

print("F1-Score without BERT:", f1_no_bert)
print("F1-Score with BERT:", f1_with_bert)

F1-Score without BERT: 0.8
F1-Score with BERT: 0.8571428571428571


In [ ]:
import spacy
from spacy.training import Example

# 加载spaCy的英语模型
nlp = spacy.load("en_core_web_sm")

# 加载UD数据集（以CoNLL-U格式）
def load_UD_data(file_path):
    """
    Load Universal Dependencies data from a CoNLL-U formatted file into spaCy
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = f.read()

    # 使用spaCy解析每个句子
    train_data = []
    for doc in nlp.pipe(data.splitlines()):
        for sent in doc.sents:
            example = Example.from_dict(sent, {
                "heads": [token.head.i for token in sent], 
                "deps": [token.dep_ for token in sent]  
            })
            train_data.append(example)
    return train_data

# 加载数据（将路径替换为你的UD数据文件路径）
train_data = load_UD_data("path_to_your_UD_data_file.conllu")

from transformers import BertTokenizer, BertModel
import torch

# 加载BERT模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 示例句子
sentence = "The cat chased a dog."

# 分词并得到BERT嵌入
inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)
outputs = model(**inputs)

# 获取BERT的输出
embeddings = outputs.last_hidden_state

# 打印BERT词嵌入的形状
print("BERT词嵌入形状:", embeddings.shape)  # [1, seq_len, 768]

import torch.nn as nn
import torch.optim as optim

# 定义神经网络模型
class DependencyParser(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DependencyParser, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

# 设置输入输出参数
input_size = 768  # BERT嵌入的维度
hidden_size = 128
output_size = 18  # 假设有18种依存关系标签（根据具体任务可以调整）

# 初始化模型
model = DependencyParser(input_size, hidden_size, output_size)

# 设置优化器和损失函数
optimizer = optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# 假设我们有BERT嵌入和真实标签，进行训练
y_true = torch.tensor([0, 1, 2, 3, 4])  # 示例标签

# 训练过程
for epoch in range(10):  # 假设我们训练10个epoch
    model.train()
    optimizer.zero_grad()

    # 获取BERT词嵌入
    sentence = "The cat chased a dog."  # 示例句子
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.squeeze(0)  # 处理嵌入维度 [seq_len, 768]

    # 模型预测
    predictions = model(embeddings)

    # 计算损失
    loss = criterion(predictions, y_true)

    # 反向传播
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

import nltk
from nltk import CFG

# 创建一个简单的句法树结构
def build_syntactic_tree(sentence):
    grammar = CFG.fromstring("""
        S -> NP VP
        NP -> Det N
        VP -> V NP
        Det -> 'the' | 'a'
        N -> 'cat' | 'dog'
        V -> 'chased'
    """)
    
    parser = nltk.ChartParser(grammar)
    for tree in parser.parse(sentence.split()):
        tree.pretty_print()

# 示例句子
sentence = "The cat chased a dog."
build_syntactic_tree(sentence)